<a href="https://colab.research.google.com/github/Ahmad-hub-bot/spendsense-ai/blob/main/notebooks/spendsense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
!pip install -q google-generativeai

In [51]:
import google.generativeai as genai

if 'api_key' not in dir():
    api_key = input("Paste your Gemini API key: ")
genai.configure(api_key=api_key)

In [52]:
# Sample SMS-style transactions — simulates what would come from real bank alerts
sample_transactions = [
    "INR 450.00 debited from your account for SWIGGY on 12-06-2026",
    "Rs 1200 spent at ZARA using your card ending 4521",
    "INR 89.50 debited for UBER ride on 13-06-2026",
    "You spent Rs 320 at MCDONALDS via UPI",
    "INR 2500.00 debited at AMAZON for online purchase",
    "Rs 150 paid to STARBUCKS via UPI",
    "INR 600 debited for OLA cab booking",
    "You spent Rs 75 at local TEA STALL via UPI",
    "INR 3200.00 debited at H&M for shopping",
    "Rs 220 paid to ZOMATO for food order",
    "INR 99 debited for METRO CARD recharge",
    "You spent Rs 1800 at DECATHLON",
    "INR 50 debited for PARKING fee",
    "Rs 4500 spent at FLIGHT BOOKING - INDIGO",
    "INR 180 paid to DOMINOS PIZZA via UPI",
    "You spent Rs 95 at LOCAL GROCERY STORE",
    "INR 6000.00 debited at MAKEMYTRIP for hotel booking",
    "Rs 60 paid for BUS TICKET via UPI",
]

# Manually labeled categories for training (this becomes your training set)
labels = [
    "food", "shopping", "travel", "food", "shopping",
    "food", "travel", "food", "shopping", "food",
    "travel", "shopping", "travel", "travel", "food",
    "daily", "travel", "travel"
]

print(f"Total samples: {len(sample_transactions)}")
print(f"Total labels: {len(labels)}")

Total samples: 18
Total labels: 18


In [53]:
import re

def parse_transaction(sms_text):
    """Extracts amount and a rough merchant guess from raw SMS-style text."""

    # Extract amount — handles "INR 450.00", "Rs 1200", "Rs. 320" etc.
    amount_match = re.search(r'(?:INR|Rs\.?)\s?([\d,]+\.?\d*)', sms_text, re.IGNORECASE)
    amount = float(amount_match.group(1).replace(',', '')) if amount_match else None

    # Extract merchant — looks for ALL-CAPS words (common in SMS for merchant names)
    merchant_match = re.findall(r'\b[A-Z]{3,}(?:\s[A-Z]{2,})*\b', sms_text)
    # Filter out common non-merchant caps words
    ignore_words = {'INR', 'RS', 'UPI', 'CARD'}
    merchants = [m for m in merchant_match if m not in ignore_words]
    merchant = merchants[0] if merchants else "UNKNOWN"

    return {"amount": amount, "merchant": merchant, "raw_text": sms_text}


# Test it on your sample data
for sms in sample_transactions[:5]:
    result = parse_transaction(sms)
    print(result)

{'amount': 450.0, 'merchant': 'SWIGGY', 'raw_text': 'INR 450.00 debited from your account for SWIGGY on 12-06-2026'}
{'amount': 1200.0, 'merchant': 'ZARA', 'raw_text': 'Rs 1200 spent at ZARA using your card ending 4521'}
{'amount': 89.5, 'merchant': 'UBER', 'raw_text': 'INR 89.50 debited for UBER ride on 13-06-2026'}
{'amount': 320.0, 'merchant': 'MCDONALDS', 'raw_text': 'You spent Rs 320 at MCDONALDS via UPI'}
{'amount': 2500.0, 'merchant': 'AMAZON', 'raw_text': 'INR 2500.00 debited at AMAZON for online purchase'}


In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Vectorize the raw SMS text into numerical features
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(sample_transactions)
y = labels

# Split into train/test (small dataset, so just a quick sanity split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Check accuracy on the held-out test set
accuracy = clf.score(X_test, y_test)
print(f"Test accuracy: {accuracy:.2f}")

# Helper function to classify new transactions
def classify_transaction(sms_text):
    X_new = vectorizer.transform([sms_text])
    return clf.predict(X_new)[0]

# Try it on a few new (unseen) examples
test_examples = [
    "Rs 500 spent at PIZZA HUT via UPI",
    "INR 3000 debited for SPICEJET flight booking",
    "Rs 80 paid for AUTO RICKSHAW",
]

for ex in test_examples:
    prediction = classify_transaction(ex)
    print(f"'{ex}' → {prediction}")

Test accuracy: 0.25
'Rs 500 spent at PIZZA HUT via UPI' → food
'INR 3000 debited for SPICEJET flight booking' → travel
'Rs 80 paid for AUTO RICKSHAW' → travel


In [55]:
import pandas as pd
from datetime import datetime, timedelta

# Simulate a week of transactions with dates (since we don't have live data yet)
# In production this would come from your live Google Sheet feed
transaction_log = [
    {"date": "2026-06-15", "merchant": "SWIGGY", "amount": 450, "category": "food"},
    {"date": "2026-06-16", "merchant": "ZOMATO", "amount": 220, "category": "food"},
    {"date": "2026-06-16", "merchant": "UBER", "amount": 89.5, "category": "travel"},
    {"date": "2026-06-17", "merchant": "MCDONALDS", "amount": 320, "category": "food"},
    {"date": "2026-06-18", "merchant": "STARBUCKS", "amount": 150, "category": "food"},
    {"date": "2026-06-18", "merchant": "ZARA", "amount": 1200, "category": "shopping"},
    {"date": "2026-06-19", "merchant": "DOMINOS", "amount": 180, "category": "food"},
    {"date": "2026-06-20", "merchant": "TEA STALL", "amount": 75, "category": "food"},
    {"date": "2026-06-21", "merchant": "OLA", "amount": 600, "category": "travel"},
]

df = pd.DataFrame(transaction_log)
df['date'] = pd.to_datetime(df['date'])

# Set weekly budgets per category (you can tune these)
budgets = {
    "food": 1500,
    "shopping": 3000,
    "travel": 1000,
    "daily": 800
}
def forecast_breach(df, category, weekly_budget, week_start, week_end, today=None):
    """Checks current spend pace and forecasts if it'll exceed budget by week end."""

    if today is None:
        today = df['date'].max()  # use latest transaction date as "today" instead of real-world now()

    week_data = df[(df['category'] == category) &
                    (df['date'] >= week_start) &
                    (df['date'] <= week_end)]

    current_spend = week_data['amount'].sum()
    days_elapsed = (today.date() - week_start.date()).days + 1
    days_elapsed = max(days_elapsed, 1)
    total_days = (week_end - week_start).days + 1

    daily_rate = current_spend / days_elapsed
    projected_total = daily_rate * total_days

    will_breach = projected_total > weekly_budget

    return {
        "category": category,
        "current_spend": round(current_spend, 2),
        "projected_total": round(projected_total, 2),
        "budget": weekly_budget,
        "will_breach": will_breach
    }

In [56]:
import requests

bot_token = input("Paste your Telegram bot token: ")
chat_id = input("Paste your chat ID: ")

def send_telegram_alert(message):
    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
    payload = {"chat_id": chat_id, "text": message}
    response = requests.post(url, data=payload)
    return response.json()

# Test it
result = send_telegram_alert("✅ SpendSense bot is connected and working!")
print(result)

Paste your Telegram bot token: 8923339991:AAEp4k7sjaF4I7isKdiVLFPksFe5fEERM08
Paste your chat ID: 8728889026
{'ok': True, 'result': {'message_id': 11, 'from': {'id': 8923339991, 'is_bot': True, 'first_name': 'spendsense_alert_bot', 'username': 'spendsense_alert_bot'}, 'chat': {'id': 8728889026, 'first_name': 'Bizzy', 'last_name': 'Doodle', 'type': 'private'}, 'date': 1782002336, 'text': '✅ SpendSense bot is connected and working!'}}


In [57]:
import requests

bot_token = input("Paste your Telegram bot token: ")
chat_id = input("Paste your chat ID: ")

def send_telegram_alert(message):
    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
    payload = {"chat_id": chat_id, "text": message}
    response = requests.post(url, data=payload)
    return response.json()

# Test it
result = send_telegram_alert("✅ SpendSense bot is connected and working!")
print(result)

Paste your Telegram bot token: 8923339991:AAEp4k7sjaF4I7isKdiVLFPksFe5fEERM08
Paste your chat ID: 8728889026
{'ok': True, 'result': {'message_id': 12, 'from': {'id': 8923339991, 'is_bot': True, 'first_name': 'spendsense_alert_bot', 'username': 'spendsense_alert_bot'}, 'chat': {'id': 8728889026, 'first_name': 'Bizzy', 'last_name': 'Doodle', 'type': 'private'}, 'date': 1782002378, 'text': '✅ SpendSense bot is connected and working!'}}


In [66]:
for cat, budget in budgets.items():
    result = forecast_breach(df, cat, budget, week_start, week_end, today=simulated_today)
    if result['will_breach']:
        alert_message = (
            f"⚠️ Budget Alert: {cat.upper()}\n"
            f"Spent so far: ₹{result['current_spend']}\n"
            f"Projected by week end: ₹{result['projected_total']}\n"
            f"Budget: ₹{result['budget']}\n"
            f"You're on pace to exceed this category's budget!"
        )
        send_telegram_alert(alert_message)
        print(f"🔔 Alert sent for {cat}")
    else:
        print(f"✅ {cat} on track — no alert needed")

🔔 Alert sent for food
✅ shopping on track — no alert needed
✅ travel on track — no alert needed
✅ daily on track — no alert needed
